In [1]:
import pandas as pd
import numpy as np

# Import scoring functions from anomaly_scores.py
from anomaly_scores import score_knn, score_lof, score_loci, score_inflo, score_cof, score_ccof, get_metrics
from anomaly_scores import make_results_table 

In [2]:

def print_latex_rows(df):
    for _, row in df.iterrows():
        print(
            f"{int(row['m'])} & "
            f"{int(row['knn_mt'])} & {row['knn_Re']:.3f} & {row['knn_RP']:.3f} & "
            f"{int(row['lof_mt'])} & {row['lof_Re']:.3f} & {row['lof_RP']:.3f} & "
            f"{int(row['loci_mt'])} & {row['loci_Re']:.3f} & {row['loci_RP']:.3f} & "
            f"{int(row['inflo_mt'])} & {row['inflo_Re']:.3f} & {row['inflo_RP']:.3f} & "
            f"{int(row['cof_mt'])} & {row['cof_Re']:.3f} & {row['cof_RP']:.3f} & "
            f"{int(row['ccof_mt'])} & {row['ccof_Re']:.3f} & {row['ccof_RP']:.3f} \\\\\n\\hline"
        )

k_values = [10, 20, 30, 40, 50]
m_values = [5, 10, 15, 20, 30]

if __name__ == "__main__":
    # Load dataset (change to dat1.txt, dat2.txt, or dat3.txt)
    #file_path = "/Users/smahmood1/Library/CloudStorage/Dropbox/MyPapers/Anomaly/dat1.txt"
    file_path = "/Users/smahmood1/Library/CloudStorage/Dropbox/MyPapers/Anomaly/dat3.txt"
    dat = pd.read_csv(file_path, sep=r'\s+')
    
    feature_cols = [c for c in dat.columns if c not in ['label']]
    X = dat[feature_cols].values
    y = np.where(dat['label'] == 'o', 1, 0)
    
    # Run evaluations across different k parameters
    for k_val in k_values:
        print(f"\n %LaTeX Table Output for k = {k_val} %")
        print(rf"\multicolumn{{1}}{{|c|}}{{${{k}}={k_val}$}} & \multicolumn{{3}}{{|c|}}{{K-NN}} & \multicolumn{{3}}{{|c|}}{{LOF}}")
        print(r"    & \multicolumn{3}{|c|}{LOCI} & \multicolumn{3}{|c|}{INFLO} & \multicolumn{3}{|c|}{COF} & \multicolumn{3}{|c|}{CCOF}\\")
        print(r"    \hline")
        print(r"    $m$ & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP \\")
        print(r"\hline ")

        tab = make_results_table(X, y, k=k_val, m_values=m_values)
        print_latex_rows(tab)


 %LaTeX Table Output for k = 10 %
\multicolumn{1}{|c|}{${k}=10$} & \multicolumn{3}{|c|}{K-NN} & \multicolumn{3}{|c|}{LOF}
    & \multicolumn{3}{|c|}{LOCI} & \multicolumn{3}{|c|}{INFLO} & \multicolumn{3}{|c|}{COF} & \multicolumn{3}{|c|}{CCOF}\\
    \hline
    $m$ & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP & $m_t$ & Re & RP \\
\hline 
5 & 5 & 0.833 & 0.997 & 5 & 0.833 & 0.997 & 5 & 0.833 & 0.997 & 5 & 0.833 & 0.997 & 5 & 0.833 & 0.997 & 3 & 0.500 & 0.995 \\
\hline
10 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.995 \\
\hline
15 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.995 \\
\hline
20 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.995 \\
\hline
30 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6 & 1.000 & 0.997 & 6

In [17]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

# Import anomaly scoring functions from your custom module
from anomaly_scores import (
    score_knn,
    score_lof,
    score_cof,
    score_loci,
    score_inflo,
    score_ccof
)

from synthetic_topology import (generate_global_local_boundary,
                                  generate_non_convex_manifolds,
                                  generate_multi_density_clusters,
                                  generate_micro_clusters)

def compute_auc_pr(y_true, y_scores):
    """Calculates Area Under the Precision-Recall Curve (AUC-PR)."""
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    return auc(recall, precision)


# =====================================================================
# 2. EVALUATION PIPELINE & LATEX TABLE BUILDER
# =====================================================================

def run_synthetic_auc_evaluation(k):
    topologies = {
        "Global \\& Local Boundary": generate_global_local_boundary(),
        "Non-Convex Manifolds": generate_non_convex_manifolds(),
        "Multi-Density Clusters": generate_multi_density_clusters(),
        "Small Micro-Clusters ($S \\le k$)": generate_micro_clusters()
    }
    
    methods = {
        "$k$-NN": score_knn,
        "LOF": score_lof,
        "LOCI": score_loci,
        "INFLO": score_inflo,
        "COF": score_cof,
        "CCOF": score_ccof
    }
    
    results = {}
    
    for top_name, (X, y) in topologies.items():
        results[top_name] = {"AUC-ROC": {}, "AUC-PR": {}}
        
        for method_name, score_fn in methods.items():
            try:
                # Compute anomaly scores
                scores = score_fn(X, k=k)
                
                # Replace any NaN/Inf with 0
                scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
                
                auc_roc = roc_auc_score(y, scores)
                auc_pr = compute_auc_pr(y, scores)
                
                results[top_name]["AUC-ROC"][method_name] = auc_roc
                results[top_name]["AUC-PR"][method_name] = auc_pr
            except Exception as e:
                results[top_name]["AUC-ROC"][method_name] = 0.0
                results[top_name]["AUC-PR"][method_name] = 0.0
                print(f"Error executing {method_name} on {top_name}: {e}")

    # Build LaTeX Table
    latex_output = []
    latex_output.append("\\begin{table*}[t]")
    latex_output.append("\\centering")
    latex_output.append("\\caption{Empirical performance comparison across synthetic benchmark topologies measured by AUC-ROC and AUC-PR. Bold values indicate superior performance.}")
    latex_output.append("\\label{tab:synthetic_auc}")
    latex_output.append("\\scalebox{0.8}{")
    latex_output.append("\\begin{tabular}{llcccccc}")
    latex_output.append("\\hline")
    latex_output.append("\\textbf{Topology} & \\textbf{Metric} & \\textbf{$k$-NN} & \\textbf{LOF} & \\textbf{COF} & \\textbf{INFLO} & \\textbf{LOCI} & \\textbf{CCOF} \\\\ \\hline")
    
    for top_idx, (top_name, metrics) in enumerate(results.items()):
        for metric_name in ["AUC-ROC", "AUC-PR"]:
            row_scores = metrics[metric_name]
            max_val = max(row_scores.values())
            
            row_str = []
            if metric_name == "AUC-ROC":
                row_str.append(f"{top_name} & {metric_name}")
            else:
                row_str.append(f" & {metric_name}")
                
            for method_name in methods.keys():
                val = row_scores[method_name]
                val_fmt = f"{val:.3f}"
                
                # Bold max value (or within floating-point epsilon)
                if abs(val - max_val) < 1e-4:
                    val_fmt = f"\\textbf{{{val_fmt}}}"
                    
                row_str.append(val_fmt)
                
            line = " & ".join(row_str) + " \\\\"
            if metric_name == "AUC-PR":
                line += " \\hline"
            latex_output.append(line)
            
    latex_output.append("\\end{tabular}%")
    latex_output.append("}")
    latex_output.append("\\end{table*}")    
    print("\n".join(latex_output))


if __name__ == "__main__":
    run_synthetic_auc_evaluation(k=40)  # change k to other values like 10, 20, 30, etc.


\begin{table*}[t]
\centering
\caption{Empirical performance comparison across synthetic benchmark topologies measured by AUC-ROC and AUC-PR. Bold values indicate superior performance.}
\label{tab:synthetic_auc}
\scalebox{0.8}{
\begin{tabular}{llcccccc}
\hline
\textbf{Topology} & \textbf{Metric} & \textbf{$k$-NN} & \textbf{LOF} & \textbf{COF} & \textbf{INFLO} & \textbf{LOCI} & \textbf{CCOF} \\ \hline
Global \& Local Boundary & AUC-ROC & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} \\
 & AUC-PR & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} & \textbf{1.000} \\ \hline
Non-Convex Manifolds & AUC-ROC & 0.911 & 0.321 & \textbf{0.982} & 0.926 & 0.942 & 0.939 \\
 & AUC-PR & 0.116 & 0.014 & \textbf{0.736} & 0.182 & 0.289 & 0.234 \\ \hline
Multi-Density Clusters & AUC-ROC & 0.988 & 0.988 & 0.978 & 0.987 & 0.974 & \textbf{0.996} \\
 & AUC-PR & 0.621 & 0.561 & 0.455 & 0.546 & 0.419 & \textbf{0.748} \\ \hli